In [ ]:
# ici nous créerons à partir des données propre de notre dataset, 
# des variables explicatives qui décrivent(de maniére non exhaustive) 
# le marché de la cryptomonnaie. Entre autres :

# Le Momentum : accélaration ou ralentissement du marché (return, return_1, return_3, return_7, rsi)
# La volatilité : la stabilité du marché (volatility_5, volatility_20, BB_width)
# la structure du marché : position du prix dans sa position technique (dist_sma, dist_ema, BB_percent,
# BB_upper, BB_middle, BB_lower)
# la tendance : la direction du marché (sma_20, ema_20, trend_signal)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
CLEAN_DATA_PATH = Path("../data/cleaned")
FEATURE_DATA_PATH = Path("../data/final")

FEATURE_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(
    CLEAN_DATA_PATH /
    "BTCUSD_clean.csv"
)

df["date"] = pd.to_datetime(df["date"])

df.head()

,symbol,date,open,high,low,close,volume,change,changePercent,vwap
0,BTCUSD,2012-04-17,4.95914,5.02207,4.92500,4.97587,0,0.01673,0.33736,4.97052
1,BTCUSD,2012-04-18,4.97587,5.17820,4.96000,5.11780,0,0.14193,2.85237,5.05797
2,BTCUSD,2012-04-19,5.11760,5.19000,5.10000,5.13766,0,0.02006,0.39198,5.13632
3,BTCUSD,2012-04-20,5.13755,5.48000,5.10103,5.35000,0,0.21245,4.13524,5.26714
4,BTCUSD,2012-04-21,5.39771,5.47900,5.11000,5.26008,0,-0.13763,-2.54979,5.31170


In [ ]:
# rendement journalier : nous permtes de savoir à quelle vitesse le marché change

df["return"] = df["close"].pct_change()

# rendement sur plusieurs périodes

df["return_1"] = df["close"].pct_change(1)

df["return_3"] = df["close"].pct_change(3)

df["return_7"] = df["close"].pct_change(7)

In [5]:
#Volatilité : niveau de bruits du marché

df["volatility_5"] = (df["return"].rolling(5).std())
# bruit du marché à court terme

df["volatility_20"] = (df["return"].rolling(20).std())
# bruit plus long

In [7]:
#Moyennes mobiles

df["sma_20"] = (df["close"].rolling(20).mean())

df["ema_20"] = (df["close"].ewm(span=20,adjust=False).mean())

In [8]:
df["dist_sma"] = ((df["close"] - df["sma_20"])/df["sma_20"])

df["dist_ema"] = ((df["close"] - df["ema_20"])/df["ema_20"])

#mesure l'écart entre la prix et la tendance.

In [9]:
#Rsi

delta = df["close"].diff()

gain = delta.clip(lower=0)

loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()

avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["rsi"] = (100 -(100/(1+rs)))

In [10]:
# bande de bollinger

rolling_std = (df["close"].rolling(20).std())

df["BB_middle"] = df["sma_20"]

df["BB_upper"] = (df["BB_middle"] + 2*rolling_std)

df["BB_lower"] = (df["BB_middle"] - 2*rolling_std)

df["BB_width"] = ((df["BB_upper"]-df["BB_lower"]) / df["BB_middle"])

df["BB_percent"] = ((df["close"]-df["BB_lower"]) / (df["BB_upper"]-df["BB_lower"]))

In [11]:
df["trend_signal"] = np.where(df["ema_20"] > df["sma_20"],1,0)

In [ ]:
df["return_acceleration"] = (df["return"].diff()) #Acceleration du marché

In [13]:
print(df.shape)

df.head()

df.info()

(5000, 28)
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   symbol               5000 non-null   str           
 1   date                 5000 non-null   datetime64[us]
 2   open                 5000 non-null   float64       
 3   high                 5000 non-null   float64       
 4   low                  5000 non-null   float64       
 5   close                5000 non-null   float64       
 6   volume               5000 non-null   int64         
 7   change               5000 non-null   float64       
 8   changePercent        5000 non-null   float64       
 9   vwap                 5000 non-null   float64       
 10  return               4999 non-null   float64       
 11  return_1             4999 non-null   float64       
 12  return_3             4997 non-null   float64       
 13  return_7             4993 non-nul

In [14]:
df.to_csv(FEATURE_DATA_PATH / "BTCUSD_features.csv",index=False)
print("Features sauvegardées.")

Features sauvegardées.
